# Dataset Analysis


* Module Name: dataset.ipynb
* Description: analysis of the Taronga Giraffe dataset

The objective of this analysis is to understand the feature distribution to ensure we have an even distribution of the features. 

Explanation of the table:
* Population: how many annotations for that class exist in the dataset
* Ratio: the ratio of annotations of this class to all classes (including this)
* Coverage: actual ratio of bounding box pixels dedicated to the feature relative to all pixels.

Copyright (C) 2025 J.Cincotta

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program. If not, see <https://www.gnu.org/licenses/>.


In [31]:
import pandas as pd
import json

In [32]:
coco_test="data/wod_reid/annotations/coco_test.json"
coco_train="data/wod_reid/annotations/coco_train.json"
coco_val="data/wod_reid/annotations/coco_val.json"


In [33]:
def extract_classes(data) -> dict:
    output: dict = {}
    for category in data["categories"]:
        output[category["id"]] = category["name"]
    return output


def get_image_id(data, image_id) -> any:
    for image in data["images"]:
        if image["id"]==image_id:
            return image
    return None


def extract_class_populations(data) -> dict:
    output: dict = {}
    for annotation in data["annotations"]:
        value: int = output.get(annotation["category_id"],0)
        value += 1
        output[annotation["category_id"]] = value
    return output


def extract_class_ratios(data) -> dict:
    output: dict = {}
    for annotation in data["annotations"]:
        image: dict = get_image_id(data, annotation["image_id"])
        width: int = int(image["width"])
        height: int = int(image["height"])
        b_width: int = int(annotation["bbox"][2])
        b_height: int = int(annotation["bbox"][3])
        value: int = output.get(annotation["category_id"],0)
        output[annotation["category_id"]] = value + ((b_width * b_height) / (width * height))
    return output


def load_data(filename):
    with open(filename, 'r') as file:
        data = json.load(file)
        return data


def analysis(data) -> pd.DataFrame:
    output: dict = {}
    classes: dict = extract_classes(data)
    populations: dict = extract_class_populations(data)
    coverage: dict = extract_class_ratios(data)
    total = sum(populations.values())
    output["class-name"] = list(classes.values())
    output["population"] = [populations[c] for c in classes.keys()]
    output["ratio"] = [f"{populations[c]/total*100:.2f}%" for c in classes.keys()]
    output["coverage"] = [f"{coverage[c]/populations[c]*100:.2f}%" for c in classes.keys()]
    df = pd.DataFrame(output)
    return df



In [34]:
print("Test Dataset")
analysis(load_data(coco_test))


Test Dataset


,class-name,population,ratio,coverage
0,Zarafa,192,27.35%,5.72%
1,Ebo,149,21.23%,5.06%
2,Jimiyu,166,23.65%,7.77%
3,Kito,195,27.78%,5.01%


In [35]:
print("Validation Dataset")
analysis(load_data(coco_val))


Validation Dataset


,class-name,population,ratio,coverage
0,Zarafa,180,25.94%,4.97%
1,Ebo,159,22.91%,3.64%
2,Jimiyu,175,25.22%,4.12%
3,Kito,180,25.94%,6.51%


In [36]:
print("Train Dataset")
analysis(load_data(coco_train))


Train Dataset


,class-name,population,ratio,coverage
0,Zarafa,1287,24.80%,5.60%
1,Ebo,1189,22.91%,4.88%
2,Jimiyu,1428,27.52%,6.16%
3,Kito,1285,24.76%,4.75%
